# Train and Predict Drones

## Introduction

This guide will show you how to train a YOLO model on a custom dataset and use it to detect drones in images/videos.

## Table of Contents

<!-- TOC -->
* [Train and Predict Drones](#train-and-predict-drones)
  * [Introduction](#introduction)
  * [Table of Contents](#table-of-contents)
* [🏁 1. Initialization](#-1-initialization)
    * [1.0 Installing dependencies](#10-installing-dependencies)
    * [1.1 Importing Libraries](#11-importing-libraries)
    * [1.2 Global Definitions](#12-global-definitions)
    * [1.3 Global Settings](#13-global-settings)
    * [1.4 Global Structure](#14-global-structure)
    * [1.5 Global Function Definitions](#15-global-function-definitions)
* [📂 2. Dataset](#-2-dataset)
    * [2.0 Acquire Dataset](#20-acquire-dataset)
    * [2.1 Split dataset](#21-split-dataset)
    * [2.2 Convert to file images](#22-convert-to-file-images)
* [📚️ 3. Settings](#-3-settings)
    * [3.1 Dataset configuration](#31-dataset-configuration)
    * [3.2 Model selection](#32-model-selection)
* [⚙️ 4. Training](#-4-training)
    * [4.1 Training Configuration](#41-training-configuration)
    * [4.2 Run Training](#42-run-training)
    * [4.3 Training results](#43-training-results)
* [✅️ 5. Validation](#-5-validation)
    * [5.1 Evaluate Model](#51-evaluate-model)
* [🏭️ 6. Inference](#-6-inference)
    * [6.0 Load model](#60-load-model)
    * [6.1 Run Predictions (IMAGES)](#61-run-predictions-images)
    * [6.2 Visualize Predictions](#62-visualize-predictions)
    * [6.3 Run Predictions (VIDEOS)](#63-run-predictions-videos)
* [⛴️ 7. Export & Deployment](#-7-export--deployment)
    * [Supported Formats](#supported-formats)
<!-- TOC -->

# 🏁 1. Initialization

### 1.0 Installing dependencies

In [1]:
import numpy as np
!pip install ultralytics==8.4.9 markdown rich wrapt pandas huggingface_hub scikit-learn opencv-python imagecodecs wandb python-dotenv datasets -q

### 1.1 Importing Libraries

In [2]:
from datasets import load_dataset, Image, concatenate_datasets, DatasetDict
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO, settings
from huggingface_hub import login
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datetime import datetime
from typing import Iterable
from pathlib import Path
from typing import Union
from wandb import Table
from tqdm import tqdm
from PIL import Image

import numpy as np
import ultralytics
import datetime
import shutil
import wandb
import torch
import uuid
import yaml
import math
import cv2
import re
import os

ultralytics.checks()

Ultralytics 8.4.9 🚀 Python-3.13.5 torch-2.10.0+cu128 CUDA:0 (NVIDIA A10, 22588MiB)
Setup complete ✅ (128 CPUs, 442.6 GB RAM, 6107.5/10076.8 GB disk)


### 1.2 Global Definitions

- **DATASET_ROOT_DIR**: Path of the dataset, directory could be empty or contain images and labels.
- **DATASET_SPLIT_NAME**: Name of the split for Hugging Face. For dataset creation it should have only one split.
- **MODELS_DIRECTORY**: Directory where models are stored.
- **LOAD_LABEL_OTHER**: Determines whether to load boxes with the label class ID "other."
    - If False, an empty label file is created for images with class "other". The model output includes only "Drone."
    - If True, the model output includes both "Drone" and "Other."

In [16]:
DATASET_ROOT_DIR = Path('./datasets/main')
DATASET_SPLIT_NAME = DATASET_ROOT_DIR / 'train_validation_test'
IMAGE_APPLY_TRANSFORMATION = "RGB"  # RGB | GRAY | SOBEL | CANNY
LOAD_LABEL_OTHER = False
DEVICE = "auto" # cuda | cpu | auto

# You may not need to modify these variables
TRAINING_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'train'
VALIDATION_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'valid'
TEST_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'test'

IMAGES_EXTENSIONS = [".jpg", ".jpeg", ".JPG", ".JPEG"]
MODELS_DIRECTORY = Path('./models/')
PROJECT_NAME = "computer-vision"
load_dotenv()

False

### 1.3 Global Settings

Log in to the different providers.

- **Hugging Face**: To download and upload dataset
- **Weights & Biases**: To keep track of the different training results

In [4]:
# YOLO settings
settings.update({"wandb": True})

# Initialize Weights & Biases environment
wandb.login(key=os.getenv("WANDB_TOKEN"))

# login("TOKEN") # Keep commented if token loaded from .env file

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: Currently logged in as: cqsv20ajs (hibou) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### 1.4 Global Structure

`dataset_structure` is used to simplify files process through the differents steps. It will be filled automatically later. You don't need to change anything.


In [5]:
dataset_structure = {
    "root": Path(""),
    "name": "",
    "classes": [],
    "img_channels": 0,
    "train": {
        "images": [],
        "labels": [],
    },
    "valid": {
        "images": [],
        "labels": [],
    },
    "test": {
        "images": [],
        "labels": [],
    },
}


### 1.5 Global Function Definitions

In [12]:
def list_files(
        directory: Union[str, Path],
        extensions: Iterable[str],
        include_root_directory: bool = False,
        recursive: bool = False,
) -> list[Path]:
    directory = Path(directory)
    extensions = tuple(extensions)

    matched_files = []

    if recursive:
        iterator = directory.rglob("*")
    else:
        iterator = directory.iterdir()

    for p in iterator:
        if p.is_file() and p.suffix in extensions:
            matched_files.append(
                p if include_root_directory else p.name
            )

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files_to_sort: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files_to_sort (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files_to_sort, key=lambda i: int(i.stem))


def parse_label(data: Union[Path, str]):
    try:
        content = data.read_text() if isinstance(data, Path) else data
        return [x for x in re.split(r"\s+", content) if x]
    except Exception as e:
        print(f"Error parsing label {data}: {e}")
        return []


def update_dataset_structure():
    dataset_structure["root"] = Path(DATASET_ROOT_DIR)
    dataset_structure["name"] = DATASET_ROOT_DIR.name

    dataset_structure["train"]["images"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["train"]["labels"] = sort_files_by_number(list_files(TRAINING_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["valid"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["valid"]["labels"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["test"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["test"]["labels"] = sort_files_by_number(list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    match IMAGE_APPLY_TRANSFORMATION:
        case "GRAY":
            dataset_structure["img_channels"] = 1
        case "RGB":
            dataset_structure["img_channels"] = 3
        case "SOBEL":
            dataset_structure["img_channels"] = 1
        case "CANNY":
            dataset_structure["img_channels"] = 1

    dataset_structure["classes"] = ["drone", "other"]


def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join(dataset_path, "backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    def ignore_backup(_, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)


def interpret_map(map_value: float) -> str:
    """
    Interpret mAP value according to standard object detection heuristics.
    """
    if map_value < 0.10:
        return "Model is effectively failing"
    elif map_value < 0.30:
        return "Very weak performance"
    elif map_value < 0.50:
        return "Usable baseline"
    elif map_value < 0.70:
        return "Good performance"
    else:
        return "Strong performance"


def plot_image_grid(images_path, nb_cols=4, max_images_preview=-1, show_title=False):
    if max_images_preview != -1:
        images_path = images_path[:max_images_preview]

    rows = math.ceil(len(images_path) / nb_cols)

    img = Image.open(images_path[0])
    w, h = img.size  # pixels

    dpi = 100

    img_w = w / dpi
    img_h = h / dpi
    #
    plt.figure(figsize=(nb_cols * img_w, rows * img_h))

    for i, path in enumerate(images_path):
        img = Image.open(path)
        plt.subplot(rows, nb_cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if show_title:
            plt.title(path.name, fontsize=25)

    plt.tight_layout()
    plt.show()


def create_run_name(version, size):
    Path("./runs").mkdir(parents=False, exist_ok=True)
    session_id = str(uuid.uuid4()).split("-")[0]

    run_name = f"{version}-{size}-{session_id}"

    with open("./runs/runs_history.txt", "a") as f:
        f.write(run_name + "\n")
    return run_name


def last_run_name(offset=0):
    with open("./runs/runs_history.txt", "r") as f:
        return f.read().splitlines()[-offset - 1]


def apply_image_transformations(image, transformation):
    """
    Apply image transformations:
    - GRAY: convert to grayscale
    - RGB: convert to RGB numpy array
    - GRAY_SOBEL: grayscale + Sobel X/Y as 2 channels
    - RGB_SOBEL: RGB + Sobel X/Y for each channel, stacked as 6 channels
    """
    # Ensure image is RGB
    np_image = np.array(image.convert("RGB"))

    match transformation:
        case "GRAY":
            gray = cv2.cvtColor(np_image, cv2.COLOR_RGB2GRAY)
            return gray  # single channel

        case "RGB":
            return cv2.cvtColor(np_image, cv2.COLOR_RGB2BGR)  # 3 channels

        case "SOBEL":
            scale = 1.5
            delta = 0
            ddepth = cv2.CV_16S

            src = cv2.cvtColor(np_image, cv2.COLOR_RGB2BGR)
            src = cv2.GaussianBlur(src, (3, 3), 0)
            gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

            grad_x = cv2.Sobel(gray, ddepth, 1, 0, ksize=3, scale=scale, delta=delta, borderType=cv2.BORDER_DEFAULT)
            grad_y = cv2.Sobel(gray, ddepth, 0, 1, ksize=3, scale=scale, delta=delta, borderType=cv2.BORDER_DEFAULT)
            #
            abs_grad_x = cv2.convertScaleAbs(grad_x)
            abs_grad_y = cv2.convertScaleAbs(grad_y)

            grad = cv2.addWeighted(abs_grad_x, 0.5, abs_grad_y, 0.5, 0)

            return grad

        case "CANNY":
            sigma = 0.1
            img = cv2.cvtColor(np_image, cv2.COLOR_RGB2BGR)
            img_blur = cv2.GaussianBlur(img, (5, 5), 0)

            v = float(np.median(img_blur))

            lower = int(max(0.0, (1.0 - sigma) * v))
            upper = int(min(255.0, (1.0 + sigma) * v))
            edges = cv2.Canny(img_blur, lower, upper)

            return edges

        case _:
            return cv2.cvtColor(np_image, cv2.IMREAD_UNCHANGED)

def get_device():
    if not torch.cuda.is_available():
        return "cpu"
    else:
        nb_gpu = torch.cuda.device_count()
        return np.arange(nb_gpu).tolist()
    return "cpu"

# 📂 2. Dataset

### 2.0 Acquire Dataset

Download the dataset from Hugging Face. [Hibou-Foundation](https://huggingface.co/Hibou-Foundation) is the official repo of the project.

If the dataset is already downloaded and converted to files, go directly to step: **3. Models Settings**

In [ ]:
dataset = load_dataset("Hibou-Foundation/computer-vision", revision="main")

### 2.1 Split dataset

Split the dataset into train, validation, test.

**Parameters**:

- **base_split**: The name of the main Hugging Face dataset split (e.g., "train_validation_test") from which the new splits will be created.
- **label_column**: In Hugging Face, the name of the column who matches class ID. For instance: 0: drone, 1: other
- **train_ratio**: A list of ratios (one per class) specifying the proportion of each class’s samples to allocate to the training set. For example, [0.8, 0.8] means 80% of samples from both class 0 and class 1 will be used for training.
- **valid_ratio**: A list of ratios (one per class) specifying the proportion of each class’s samples to allocate to the validation set. For example, [0.1, 0.1] means 10% of samples from both class 0 and class 1 will be used for validation.
- **test_ratio**: A list of ratios (one per class) specifying the proportion of each class’s samples to allocate to the test set. For example, [0.1, 0.1] means 10% of samples from both class 0 and class 1 will be used for testing.
- **seed**: The random seed used for shuffling the dataset before splitting, ensuring reproducibility of the splits.

In [ ]:
base_split = "train_validation_test"
label_column = "class_id"

train_ratio = [0.8, 0.6]  # [class 0, class 1]
valid_ratio = [0.1, 0.2]
test_ratio = [0.1, 0.2]

seed = 42

for i in range(len(train_ratio)):
    assert train_ratio[i] + valid_ratio[i] + test_ratio[i] == 1.0

train_parts = []
valid_parts = []
test_parts = []

num_classes = len(train_ratio)

for cls in range(num_classes):
    cls_ds = dataset[base_split].filter(
        lambda x: x[label_column] == cls
    )

    cls_ds = cls_ds.shuffle(seed=seed)

    n = len(cls_ds)
    n_train = int(n * train_ratio[cls])
    n_valid = int(n * valid_ratio[cls])

    train_parts.append(cls_ds.select(range(0, n_train)))
    valid_parts.append(cls_ds.select(range(n_train, n_train + n_valid)))
    test_parts.append(cls_ds.select(range(n_train + n_valid, n)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=seed)
valid_ds = concatenate_datasets(valid_parts).shuffle(seed=seed)
test_ds = concatenate_datasets(test_parts).shuffle(seed=seed)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})
dataset

### 2.2 Convert to Yolo

Now images and labels must be converted into regular files to be processed by YOLO

In [ ]:
# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)

def export_to_yolo(ds, split_name, image_transform):
    """
    Export dataset to YOLO format.
    Supports standard RGB/GRAY images and multi-channel TIFFs (e.g., RGB + Sobel 6 channels).
    """
    for idx, sample in enumerate(tqdm(ds, total=len(ds))):
        image = sample["image"]  # PIL.Image
        label = sample["raw_label"]  # YOLO format [[class, cx, cy, w, h], ...]
        img_name = sample["name"]
        txt_name = img_name.split(".")[0] + ".txt"

        label_formated = parse_label(label)

        # Skip empty labels if required
        if len(label_formated) == 1 and LOAD_LABEL_OTHER:
            continue

        # Apply transformation
        cv2_img = apply_image_transformations(image, image_transform)

        base_name = os.path.splitext(img_name)[0]
        img_path = os.path.join(DATASET_ROOT_DIR, split_name, f"{base_name}.tiff")
        cv2.imwrite(img_path, cv2_img)

        # Save YOLO labels
        lbl_path = os.path.join(DATASET_ROOT_DIR, split_name, txt_name)
        with open(lbl_path, "w") as f:
            if not LOAD_LABEL_OTHER and int(label_formated[0]) == 1:
                continue  # create an empty label file
            else:
                f.write(label)


# Run export
split_mapping = {"train": "train", "validation": "valid", "test": "test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name, IMAGE_APPLY_TRANSFORMATION)
update_dataset_structure()

# 📚️ 3. Settings

### 3.1 Dataset configuration

Save YOLO dataset configuration into `data.yaml`

In [ ]:
data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    test=os.path.join('../../', TEST_DATASET_DIRECTORY),
    nc=2 if LOAD_LABEL_OTHER else 1,
    channels=dataset_structure["img_channels"],
    names=['drone', 'other'] if LOAD_LABEL_OTHER else ['drone'],
)

data_config_path = DATASET_ROOT_DIR / 'data.yaml'

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)
%cat "$data_config_path"

### 3.2 Model selection

Select the size and the version of the YOLO model to train.

Available Sizes


| Size    | Ideal Use Case                                      |
|---------|-----------------------------------------------------|
| nano    | Resource-constrained devices, high FPS requirements |
| small   | Laptops, edge devices with moderate compute         |
| medium  | Desktop systems, accuracy-focused applications      |
| large   | High-end systems, accuracy-critical applications    |
| x-large | Server/workstation deployment, offline processing   |

**Source**: https://deepwiki.com/niconielsen32/YOLO-3D/4.2-model-size-selection

In [ ]:
selected_size = "nano"
selected_version = "26"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}
model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = MODELS_DIRECTORY / model_name
model = YOLO(model_path, task="detect")

create_run_name(selected_version,
                selected_size)  # Save the run name into a file, so it can be retrieve even after jupyter kernel ended.

# ⚙️ 4. Training

### 4.1 Training Configuration
Define hyperparameters (epochs, batch size, image size)

For more information: https://docs.ultralytics.com/usage/cfg/

In [17]:
train_config = {
    'epochs': 1,
    'imgsz': 640,
    'batch': 16,
    'warmup_epochs': 3,
    'momentum': 0.9,
    'lr0': 0.0003,
    'lrf': 0.01,
    'patience': 40,
    'optimizer': 'adamW',
    'cache': False,
    'multi_scale': 0.25,

    # Augmentation Settings
    'degrees': 5,
    'perspective': 0.0002,
    'fliplr': 0.3,
    'shear': 5,
    'scale': 0.3,
    'mosaic': 0.3,
    'close_mosaic': 50,
    'cutmix': 0.3,

    'project': PROJECT_NAME,
    'device': get_device() if DEVICE == "auto" else DEVICE,
}
train_config

{'epochs': 1,
 'imgsz': 640,
 'batch': 16,
 'warmup_epochs': 3,
 'momentum': 0.9,
 'lr0': 0.0003,
 'lrf': 0.01,
 'patience': 40,
 'optimizer': 'adamW',
 'cache': False,
 'multi_scale': 0.25,
 'degrees': 5,
 'perspective': 0.0002,
 'fliplr': 0.3,
 'shear': 5,
 'scale': 0.3,
 'mosaic': 0.3,
 'close_mosaic': 50,
 'cutmix': 0.3,
 'project': 'computer-vision',
 'device': [0, 1]}

### 4.2 Run Training

In [ ]:
model.train(
    **train_config,
    data=data_config_path,
    name="train_" + last_run_name()
)

### 4.3 Training results

Print the results generated by the training.

`last_run_name()` Retrieve dynamically the latest run name from the file, so even if the kernel has been shut down after the training, it can be executed safely.

In [ ]:
result_dir = Path("./runs", "detect", PROJECT_NAME, "train_" + last_run_name())
display(IPyImage(filename=str(result_dir / "results.png")))

model_path = result_dir / "weights" / "best.pt"
model = YOLO(model_path, task="detect")

In [ ]:
%matplotlib inline

# Retrieve val_batch images
val_batch = []
i = 0
batch_file = result_dir / f"val_batch{i}_labels.jpg"
while batch_file.exists():
    val_batch.append(batch_file)
    val_batch.append(result_dir / f"val_batch{i}_pred.jpg")
    i += 1
    batch_file = result_dir / f"val_batch{i}_labels.jpg"

# Retrieve confusion matrix images
confusion_matrix_path = [
    result_dir / "confusion_matrix.png",
    result_dir / "confusion_matrix_normalized.png"
]

# Retrieve metric images
boxes_path = [
    result_dir / "BoxF1_curve.png",
    result_dir / "BoxP_curve.png",
    result_dir / "BoxPR_curve.png",
    result_dir / "BoxR_curve.png",
]

# Show images
plot_image_grid(val_batch, nb_cols=2, show_title=True)
plot_image_grid(confusion_matrix_path, nb_cols=2)
plot_image_grid(boxes_path, nb_cols=2)


# ✅️ 5. Validation

In [ ]:
update_dataset_structure()
model_path, run_name = Path("./runs", "detect", PROJECT_NAME, "train_" + last_run_name(),
                            "weights/best.pt"), last_run_name()
# model_path, run_name = Path(MODELS_DIRECTORY, "26-medium-f90c705f.pt"), "26-medium-f90c705f"

val_model = YOLO(model_path, task="detect")
wandb_run = wandb.init(project=PROJECT_NAME, name="val_" + run_name, resume="allow", reinit=True)

In [ ]:
val_config = {
    "visualize": True,
    "save": True,
    "split": "test",
    'project': PROJECT_NAME,
    'device': get_device() if DEVICE == "auto" else DEVICE,
}

###  5.1 Evaluate Model

In [ ]:
metrics = val_model.val(**val_config, data=data_config_path, name=run_name)

In [ ]:
result_dir_val = metrics.save_dir
f1_curve = metrics.box.f1_curve[0]
conf_curve = metrics.box.px

best_idx = f1_curve.argmax()
best_f1 = f1_curve[best_idx]
best_conf = conf_curve[best_idx]

score = metrics.box.map

print(f"mAP50-95: {score:.2f}")
print(f"Best F1: {best_f1:.2f}")
print(f"Confidence threshold: {best_conf:.2f}")

Upload results to WanDB

In [ ]:
wandb.log({
    "model_name": run_name,
    "map50_95": metrics.box.map,
    "map50": metrics.box.map50,
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "best_f1": f1_curve[best_idx],
    "best_conf": conf_curve[best_idx],
})
wandb.finish()

Diplay validation results

In [ ]:
%matplotlib inline

# Retrieve val_batch images
val_batch = []
i = 0
batch_file = result_dir_val / f"val_batch{i}_labels.jpg"
while batch_file.exists():
    val_batch.append(batch_file)
    val_batch.append(result_dir_val / f"val_batch{i}_pred.jpg")
    i += 1
    batch_file = result_dir_val / f"val_batch{i}_labels.jpg"

# Retrieve confusion matrix images
confusion_matrix_path = [
    result_dir_val / "confusion_matrix.png",
    result_dir_val / "confusion_matrix_normalized.png"
]

# Retrieve metric images
boxes_path = [
    result_dir_val / "BoxF1_curve.png",
    result_dir_val / "BoxP_curve.png",
    result_dir_val / "BoxPR_curve.png",
    result_dir_val / "BoxR_curve.png",
]

# Show images
plot_image_grid(val_batch, nb_cols=2, show_title=True)
plot_image_grid(confusion_matrix_path, nb_cols=2)
plot_image_grid(boxes_path, nb_cols=2)


# 🏭️ 6. Inference

### 6.0 Load model

In [ ]:
model_path, run_name = Path("./runs", "detect", PROJECT_NAME, last_run_name(), "weights/best.pt"), last_run_name()
# model_path, run_name = Path(MODELS_DIRECTORY, "26-medium-da32db782.pt"), "26-medium-da32db782"
# model_path, run_name = Path(MODELS_DIRECTORY, "yolo11n_drone.pt"), "yolo11n_drone"

inf_model = YOLO(model_path)

### 6.1 Run Predictions (IMAGES)

In [ ]:
predictions = inf_model.predict(
    TEST_DATASET_DIRECTORY,
    save=True,
    project=Path('computer-vision', run_name),
    conf=0.25
)
predictions_output_dir = predictions[0].save_dir

### 6.2 Visualize Predictions

In [ ]:
predictions_paths = list_files(predictions_output_dir, IMAGES_EXTENSIONS, True)

plot_image_grid(predictions_paths, max_images_preview=100)

### 6.3 Run Predictions (VIDEOS)

In [ ]:
result = inf_model.track(
    source="",
    tracker="./trackers/bytetrack.yaml",
    conf=0.55,
    persist=True,
    iou=0.3,
    show=False,
    imgsz=640,
    save=True,
    project=Path('computer-vision', run_name),
    exist_ok=True
)

In [ ]:
# Source - https://stackoverflow.com/a/68994194
# Posted by Aashutosh Soni
# Retrieved 2026-02-02, License - CC BY-SA 4.0

cap1 = cv2.VideoCapture(
    '')
cap2 = cv2.VideoCapture(
    '')

fps1 = cap1.get(cv2.CAP_PROP_FPS)
fps2 = cap2.get(cv2.CAP_PROP_FPS)
fps = min(fps1, fps2)
delay = int(1000 / fps)
while cap1.isOpened() and cap2.isOpened():

    okay1, frame1 = cap1.read()
    okay2, frame2 = cap2.read()

    if not okay1 or not okay2:
        print('Cant read the video, Exit!')
        break

    cv2.imshow('real', frame1)
    cv2.imshow('fake', frame2)

    if cv2.waitKey(int(delay / 10)) & 0xFF == ord('q'):
        break

cap1.release()
cap2.release()
cv2.destroyAllWindows()


# ⛴️ 7. Export & Deployment

### Supported Formats

- [Source](https://docs.ultralytics.com/modes/export/#arguments)

| Format                                                                    | `format` Argument |
|---------------------------------------------------------------------------|-------------------|
| [PyTorch](https://pytorch.org/)                                           | -                 |
| [TorchScript](https://docs.ultralytics.com/integrations/torchscript)      | `torchscript`     |
| [ONNX](https://docs.ultralytics.com/integrations/onnx)                    | `onnx`            |
| [OpenVINO](https://docs.ultralytics.com/integrations/openvino)            | `openvino`        |
| [TensorRT](https://docs.ultralytics.com/integrations/tensorrt)            | `engine`          |
| [CoreML](https://docs.ultralytics.com/integrations/coreml)                | `coreml`          |
| [TF SavedModel](https://docs.ultralytics.com/integrations/tf-savedmodel)  | `saved_model`     |
| [TF GraphDef](https://docs.ultralytics.com/integrations/tf-graphdef)      | `pb`              |
| [TF Lite](https://docs.ultralytics.com/integrations/tflite)               | `tflite`          |
| [TF Edge TPU](https://docs.ultralytics.com/integrations/edge-tpu)         | `edgetpu`         |
| [TF.js](https://docs.ultralytics.com/integrations/tfjs)                   | `tfjs`            |
| [PaddlePaddle](https://docs.ultralytics.com/integrations/paddlepaddle)    | `paddle`          |
| [MNN](https://docs.ultralytics.com/integrations/mnn)                      | `mnn`             |
| [NCNN](https://docs.ultralytics.com/integrations/ncnn)                    | `ncnn`            |
| [IMX500](https://docs.ultralytics.com/integrations/sony-imx500){{ tip3 }} | `imx`             |
| [RKNN](https://docs.ultralytics.com/integrations/rockchip-rknn)           | `rknn`            |
| [ExecuTorch](https://docs.ultralytics.com/integrations/executorch)        | `executorch`      |
| [Axelera](https://docs.ultralytics.com/integrations/axelera)              | `axelera`         |

In [ ]:
model.export(format="onnx")